In [ ]:
from IPython.core.display import display, HTML
display(HTML('<style>.container {with:98% !important;}</style>'))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import numpy as np
import pickle
import warnings
import itertools
import glob

warnings.filterwarnings('ignore')

from scipy import signal, interpolate, stats
import seaborn as sns
from tqdm import tqdm

import psycopg2 as pyc

import shap

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,precision_score,recall_score,f1_score
from sklearn.feature_selection import RFE, RFECV
from sklearn.cluster import KMeans
from sklearn.utils import shuffle
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import make_scorer,log_loss,accuracy_score, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

from sklearn.tree import _tree
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import LassoCV,Lasso
from sklearn.tree import export_graphviz
from sklearn.preprocessing import StandardScaler


In [ ]:
feature_selected = [
    'SBP_trend_z_value',
    'SBP_trend_slope',
    'SBP_trend_start_value',
    'SBP_trend_end_value',
    'SBP_min_value',
    'SBP_max_value',
    'SBP_mean_value',
    'SBP_var_value',
    'SBP_reference_value',
    'SBP_trend_time',
    'SPO2_trend_z_value',
    'SPO2_trend_slope',
    'SPO2_trend_start_value',
    'SPO2_trend_end_value',
    'SPO2_min_value',
    'SPO2_max_value',
    'SPO2_mean_value',
    'SPO2_var_value',
    'SPO2_reference_value',
    'SPO2_trend_time',
    'HR_trend_z_value',
    'HR_trend_slope',
    'HR_trend_start_value',
    'HR_trend_end_value',
    'HR_min_value',
    'HR_max_value',
    'HR_mean_value',
    'HR_var_value',
    'HR_reference_value',
    'HR_trend_time',
    'RR_trend_z_value',
    'RR_trend_slope',
    'RR_trend_start_value',
    'RR_trend_end_value',
    'RR_min_value',
    'RR_max_value',
    'RR_mean_value',
    'RR_var_value',
    'RR_reference_value',
    'RR_trend_time',
]

In [ ]:
# load rf model
rf_class = RandomForestClassifier(n_estimators=100,random_state = 42, max_depth =5, min_samples_split =4,min_samples_leaf=2,n_jobs=4)
rf_class.fit(x_train,y_train)

In [ ]:
def extract_rules(tree, feature_names):
    tree_ = tree.tree_
    feature_name = [feature_names[i] if i !=-2 else"undefined!" for i in tree_.feature]

    def recurse(node,depth, rule):
        if tree_.feature[node]!=-2:
            name = feature_name[node]
            threshold = tree_.threshold[node]
            rule.append(f"{name} <= {np.round(threshold,3)}")
            recurse(tree_.children_left[node],depth +1 ,rule)
            rule.pop()
            rule.append(f"{name} > {np.round(threshold,3)}")
            recurse(tree_.children_right[node],depth +1 ,rule)
            rule.pop()
        else:
            rules.append(" and ".join(rule))
    
    rules=[]
    recurse(0, 1, [])
    return rules

# convert tree to rules
all_rules = []
for tree in rf_class.estimators_:
    all_rules.extend(extract_rules(tree,[f'feature_{i}' for i in range(x_train.shape[1])]))
print(len(all_rules))

def evalute_rule(rule,X):
    conditions = rule.split(" and ")
    result = np.ones(X.shape[0],dtype = bool)
    for condition in conditions:
        feature, operator, threshold = condition.split()
        feature_idx = int(feature.split("_")[1])
        threshold = float(threshold)
        if operator == '<=':
            result &=(X[:,feature_idx] <= threshold)
        elif operator == ">":
            result &=(X[:,feature_idx] > threshold)
    return result


In [ ]:
# lasso select rule
x_rule_batches = []
batch_size =20
n_batches = int(np.ceil(x_train.shape[0]/batch_size))
lasso = Lasso(alpha = 0.0014)
# 迭代
# all_rules = selected_rules
x_rules = np.array([evalute_rule(rule,x_new_train) for rule in all_rules]).T
coef_sum = np.zeros(x_rules.shape[1])
for i in range(n_batches):
    start_idx = i* batch_size
    end_idx = min((i+1)*batch_size,x_rules.shape[0])
    x_batch = x_rules[start_idx:end_idx]
    y_batch = y_train[start_idx:end_idx]
    lasso.fit(x_batch,y_batch)
    coef_sum+=lasso.coef_

coef_avg = coef_sum / (x_rules.shape[0]/batch_size)
# lasso.fit(x_rules,y_train)
selected_rules_indices = np.where(coef_avg !=0)[0]
selected_rules = [all_rules[i] for i in selected_rules_indices]
print(len(selected_rules))
for rule in selected_rules:
    print(rule)

In [ ]:
# output rules
output_rule = selected_rules
rename_rule = []
for rule in output_rule:
    for i,feature in enumerate(feature_selected):
#         print(i)
        placeholder = f'feature_{i} '
#         print(placeholder)
#         print(feature)
        rule = rule.replace(placeholder, feature)
    rename_rule.append(rule)
    print (rule)